# Data Extraction

In [1]:
import requests
from bs4 import BeautifulSoup
import argparse
from urllib.parse import urlparse
import subprocess
import os
import tempfile
import json
import sys
import pickle
import pandas as pd
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
df = pd.DataFrame()
with open("../results/results_CI_v3.pkl", 'rb') as f:
    df = pickle.load(f)

In [3]:
df.head(2)

,origin,revision,branch,snapshot_without,path,status,is_ci_file
0,https://github.com/samvarankashyap/operator-test-playbooks,swh:1:rev:b029e6fe6516142bf8684040bfdf7184557de427,refs/heads/update_midstream_endpoint,swh:1:snp:08a35d363c2fcd21166fe24d878e3d25941acab1,./.github/workflows/build_and_push_latest_image.yml,Modified,1
1,https://github.com/samvarankashyap/operator-test-playbooks,swh:1:rev:b029e6fe6516142bf8684040bfdf7184557de427,refs/heads/update_midstream_endpoint,swh:1:snp:08a35d363c2fcd21166fe24d878e3d25941acab1,./.github/workflows/build_and_push_image_on_tag.yml,Modified,1


In [4]:
print(f'amount of CI files rewrites in the history when they\'re the only target: {len(df):,}')

amount of CI files rewrites in the history when they're the only target: 9,057


In [5]:
main_branch = df.loc[df['branch'].apply(lambda x: "refs/heads/main" in x or "refs/heads/master" in x)]
print(f'amount of CI files rewrites in the history in the main branch when they\'re the only target: {len(main_branch):,}')

amount of CI files rewrites in the history in the main branch when they're the only target: 481


In [6]:
main_branch.head()

,origin,revision,branch,snapshot_without,path,status,is_ci_file
3,https://github.com/chetabahana/ace-builds,swh:1:rev:b72c6b917253c99faab35cd7b80adb15dd1cdc88,refs/heads/master,swh:1:snp:672180e12f8472ee189938a117a72bd166531769,./.github/workflows/publish.yml,Modified,1
7,https://github.com/ryancdotorg/VerifyJS,swh:1:rev:8c29c5b12229e65ef6521a0cbc192171ff585be2,refs/heads/master,swh:1:snp:37bc9b93897dea2ea6249bafbf0b1f20d52097ae,./.github/workflows/main.yml,Modified,1
8,https://github.com/nir9/noteserver,swh:1:rev:430ae72e2faaf93229bbda377466ed7fc4166d49,refs/heads/master,swh:1:snp:bc84cb7f0f686eac93ad45954635722e9354726b,./.github/workflows/build.yml,Modified,1
47,https://gitlab.com/jacana-security-research/jacana.io.git,swh:1:rev:0e5ebc771c6a56ac8874df7f899839a848e361a9,refs/heads/main,swh:1:snp:08b35396a613e39f80c5eb31ad8c29d477543779,./.gitlab-ci.yml,Modified,1
70,https://github.com/Seeed-Studio/Seeed_SHT35,swh:1:rev:4a3085700bebc7b9de25ccfbc43bb7df0acef726,refs/heads/master,swh:1:snp:de824749b2c7cb6c7f560bc18aabaa831ae307e7,./.travis.yml,Modified,1


In [7]:
from swh_license_files import generate_swh_url

In [8]:
df['Rev-CI-Path'] = df.apply(
    lambda x: generate_swh_url(x['revision'], x['path']),
    axis=1
)

In [9]:
df['Snap-CI-Path'] = df.apply(
    lambda row: generate_swh_url(row['snapshot_without'], row['path'], row['branch']),
    axis=1
)

In [11]:
df.loc[df['status'].apply(lambda x: x != "Modified")]

,origin,revision,branch,snapshot_without,path,status,is_ci_file,Rev-CI-Path,Snap-CI-Path
5,https://github.com/eserte/Geo-Coder-Googlev3,swh:1:rev:3bc57b03e212d28e8d9900326fa0538f377d73a6,refs/heads/XXX-appveyor,swh:1:snp:f85af755ca3666635a1f44b93c3c54af35181810,./.travis.yml,NotFound,1,https://archive.softwareheritage.org/browse/revision/3bc57b03e212d28e8d9900326fa0538f377d73a6/?path=.travis.yml,https://archive.softwareheritage.org/browse/snapshot/f85af755ca3666635a1f44b93c3c54af35181810/directory/?branch=refs/heads/XXX-appveyor&path=.travis.yml
71,https://github.com/infusionsoft/wordpress-plugin,swh:1:rev:d924c208d978935118f36b11a134eed5253c6187,refs/heads/copyvars,swh:1:snp:ce2696536ba5fb4e05864e335c757cafad821cad,./.circleci/config.yml,NotFound,1,https://archive.softwareheritage.org/browse/revision/d924c208d978935118f36b11a134eed5253c6187/?path=.circleci/config.yml,https://archive.softwareheritage.org/browse/snapshot/ce2696536ba5fb4e05864e335c757cafad821cad/directory/?branch=refs/heads/copyvars&path=.circleci/config.yml
79,https://github.com/moker59/CreativeGates,swh:1:rev:8e075fbeebd7a0eb37f9815849fb4ac85ced97ea,refs/heads/master,swh:1:snp:4c903435a5b7802858abeb5cc694bfd081920d83,./.travis.yml,NotFound,1,https://archive.softwareheritage.org/browse/revision/8e075fbeebd7a0eb37f9815849fb4ac85ced97ea/?path=.travis.yml,https://archive.softwareheritage.org/browse/snapshot/4c903435a5b7802858abeb5cc694bfd081920d83/directory/?branch=refs/heads/master&path=.travis.yml
166,https://github.com/jeanetienne/Bee,swh:1:rev:409803ae0935a3af4b20b921129b113c9b71c6ec,refs/heads/master,swh:1:snp:064ecafaf93bb31bb7013e30f507046cb425a65f,./.circleci/config.yml,NotFound,1,https://archive.softwareheritage.org/browse/revision/409803ae0935a3af4b20b921129b113c9b71c6ec/?path=.circleci/config.yml,https://archive.softwareheritage.org/browse/snapshot/064ecafaf93bb31bb7013e30f507046cb425a65f/directory/?branch=refs/heads/master&path=.circleci/config.yml
248,https://github.com/Ichimonji10/pulp-ci,swh:1:rev:eb7787098ba8124299dc035469b6dc3f4de3264e,refs/heads/inventory,swh:1:snp:a909955cd0e17854f7af109a418047a402db52ae,./ci/jjb/jobs/pulp-3-installer.yaml,NotFound,1,https://archive.softwareheritage.org/browse/revision/eb7787098ba8124299dc035469b6dc3f4de3264e/?path=ci/jjb/jobs/pulp-3-installer.yaml,https://archive.softwareheritage.org/browse/snapshot/a909955cd0e17854f7af109a418047a402db52ae/directory/?branch=refs/heads/inventory&path=ci/jjb/jobs/pulp-3-installer.yaml
...,...,...,...,...,...,...,...,...,...
8598,https://github.com/christiaan/InlineStyle,swh:1:rev:27a103bda362c7430aee12e3aa8a90a9fd5180ec,HEAD,swh:1:snp:f4312de8e04e693ca19a6a4b935f338353b0d72a,./ci/docker_install.sh,NotFound,1,https://archive.softwareheritage.org/browse/revision/27a103bda362c7430aee12e3aa8a90a9fd5180ec/?path=ci/docker_install.sh,https://archive.softwareheritage.org/browse/snapshot/f4312de8e04e693ca19a6a4b935f338353b0d72a/directory/?branch=HEAD&path=ci/docker_install.sh
8630,https://github.com/LeoHsiao1/pyexiv2,swh:1:rev:a24f5748cc9c43acd2ffdfa477526de88739f515,refs/heads/dev,swh:1:snp:1b3f2bfe18acace09e94d2d814b749c347f8aef7,./.github/workflows/build_exiv2.yml,NotFound,1,https://archive.softwareheritage.org/browse/revision/a24f5748cc9c43acd2ffdfa477526de88739f515/?path=.github/workflows/build_exiv2.yml,https://archive.softwareheritage.org/browse/snapshot/1b3f2bfe18acace09e94d2d814b749c347f8aef7/directory/?branch=refs/heads/dev&path=.github/workflows/build_exiv2.yml
8667,https://github.com/centic9/tellico-ppa,swh:1:rev:07cc40a767c92b7bdeb455546b0b9e460c0d8b5e,refs/heads/ppa/3.1.3-1ubuntu1,swh:1:snp:76a74b03b990d423395d7e053bc170526132b27c,./.travis.yml,NotFound,1,https://archive.softwareheritage.org/browse/revision/07cc40a767c92b7bdeb455546b0b9e460c0d8b5e/?path=.travis.yml,https://archive.softwareheritage.org/browse/snapshot/76a74b03b990d423395d7e053bc170526132b27c/directory/?branch=refs/heads/ppa/3.1.3-1ubuntu1&path=.travis.yml
8732,https://github.com/EduardoAmaral/avalon,swh:1:rev

In [11]:
with open("../results/CI_v3_with_paths.pkl", 'wb') as f:
    pickle.dump(df, f)

In [13]:
with open("../results/teaser_CI_v3_with_paths.pkl", 'wb') as f:
    pickle.dump(df.head(2), f)